In [28]:
import logging
from typing import Any, Dict
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, slice, arrays_zip, explode, current_timestamp, lit, to_timestamp, from_unixtime
from pyspark.sql.types import StructType
from src.etls.bronze.load_json import LoadedJSON, load_json_data
from src.etls.utils.spark import get_spark, build_spark_table
from src.etls.gold.silver2gold import TransformationPowerDataGold
from src.etls.utils.schema import bronze_schema
import os

In [29]:
from src.etls import etl_workflow

In [32]:
!python -m src.etls.etl_workflow

IOStream.flush timed out
26/01/11 09:45:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2026-01-11 09:45:29,983 - INFO - Starting ETL Energy Data Collection...
2026-01-11 09:45:29,983 - INFO - storage path is for test data_storage/bronze/
2026-01-11 09:45:29,983 - INFO - Fetching https://api.energy-charts.info/price?bzn=DE-LU (attempt 1)
2026-01-11 09:45:30,233 - INFO - Saving the json file....
2026-01-11 09:45:30,237 - INFO - [SAVED]: /app/data_storage/bronze/6556ec96__2026-01-11-09-45-29.json
2026-01-11 09:45:30,237 - INFO - Saving the json file....
2026-01-11 09:45:30,238 - INFO - [SAVED]: /app/data_storage/bronze/meta_data/6556ec96__2026-01-11-09-45-29.meta.json
2026-01-11 09:45:30,238 - INFO - E

In [22]:
!python -m src.etls.etl_workflow

IOStream.flush timed out
2026-01-10 14:15:15,159 - INFO - Starting ETL Energy Data Collection...
2026-01-10 14:15:15,159 - INFO - storage path is for test data_storage/bronze/
2026-01-10 14:15:15,159 - INFO - Fetching https://api.energy-charts.info/price?bzn=DE-LUddd (attempt 1)
2026-01-10 14:15:15,270 - WARNING - Attempt 1 failed: 400 Client Error: Bad Request for url: https://api.energy-charts.info/price?bzn=DE-LUddd
2026-01-10 14:15:15,270 - INFO - Fetching https://api.energy-charts.info/price?bzn=DE-LUddd (attempt 2)
2026-01-10 14:15:15,398 - WARNING - Attempt 2 failed: 400 Client Error: Bad Request for url: https://api.energy-charts.info/price?bzn=DE-LUddd
2026-01-10 14:15:15,399 - INFO - Fetching https://api.energy-charts.info/price?bzn=DE-LUddd (attempt 3)
2026-01-10 14:15:15,515 - WARNING - Attempt 3 failed: 400 Client Error: Bad Request for url: https://api.energy-charts.info/price?bzn=DE-LUddd
2026-01-10 14:15:15,515 - INFO - Saving the meta data..
2026-01-10 14:15:15,515 - I

In [4]:
!python -m src.etls.silver.bronze2silver

26/01/09 12:25:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Data written to PostgreSQL table 'energy_price_silver' successfully.            


In [7]:
!python -m src.etls.gold.silver2gold

IOStream.flush timed out
26/01/09 14:49:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/09 14:49:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
Data read from PostgreSQL table 'energy_price_silver' successfully.
Data written to PostgreSQL table 'energy_price_gold' successfully.
26/01/09 14:49:27 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-1fc24a15-c05d-4b9f-83ab-cc65ee2bddc0. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-1fc24a15-c05d-4b9f-83ab-cc65ee2bddc0
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:199)
	at org.apache.spark.

In [ ]:

from src.etls.utils.spark import get_spark, read_from_postgres

# Initialize SparkSession
spark = get_spark()


# Read table into Spark DataFrame
table_name = "energy_price_gold"
df = read_from_postgres(spark=spark, table_name=table_name)

# Show top rows
df.show()


Data read from PostgreSQL table 'energy_price_gold' successfully.
+-------------------+---------+-----+-------------------+
|           req_time|     unit|price|          timePrice|
+-------------------+---------+-----+-------------------+
|2026-01-09 12:17:04|EUR / MWh|95.56|2026-01-08 23:00:00|
|2026-01-09 12:17:04|EUR / MWh|80.23|2026-01-08 23:15:00|
|2026-01-09 12:17:04|EUR / MWh| 70.1|2026-01-08 23:30:00|
|2026-01-09 12:17:04|EUR / MWh|69.96|2026-01-08 23:45:00|
|2026-01-09 12:17:04|EUR / MWh|75.48|2026-01-09 00:00:00|
|2026-01-09 12:17:04|EUR / MWh|74.24|2026-01-09 00:15:00|
|2026-01-09 12:17:04|EUR / MWh|69.98|2026-01-09 00:30:00|
|2026-01-09 12:17:04|EUR / MWh|63.06|2026-01-09 00:45:00|
|2026-01-09 12:17:04|EUR / MWh|64.21|2026-01-09 01:00:00|
|2026-01-09 12:17:04|EUR / MWh|58.11|2026-01-09 01:15:00|
|2026-01-09 12:17:04|EUR / MWh|54.79|2026-01-09 01:30:00|
|2026-01-09 12:17:04|EUR / MWh|52.29|2026-01-09 01:45:00|
|2026-01-09 12:17:04|EUR / MWh|55.75|2026-01-09 02:00:00|
|2026-

2026-01-10 16:52:25,125 - INFO - Error while sending or receiving.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/py4j/clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
2026-01-10 16:52:25,126 - INFO - Closing down clientserver connection
2026-01-10 16:52:25,126 - INFO - Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/py4j/clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local

In [15]:
df.withColumn(
    "timePrice",
    from_unixtime(col("unix_seconds")).cast("timestamp")
).select(
    "req_time",
    "unit",
    "price",
    "timePrice"
).show()


+-------------------+---------+-----+-------------------+
|           req_time|     unit|price|          timePrice|
+-------------------+---------+-----+-------------------+
|2026-01-09 12:17:04|EUR / MWh|95.56|2026-01-08 23:00:00|
|2026-01-09 12:17:04|EUR / MWh|80.23|2026-01-08 23:15:00|
|2026-01-09 12:17:04|EUR / MWh| 70.1|2026-01-08 23:30:00|
|2026-01-09 12:17:04|EUR / MWh|69.96|2026-01-08 23:45:00|
|2026-01-09 12:17:04|EUR / MWh|75.48|2026-01-09 00:00:00|
|2026-01-09 12:17:04|EUR / MWh|74.24|2026-01-09 00:15:00|
|2026-01-09 12:17:04|EUR / MWh|69.98|2026-01-09 00:30:00|
|2026-01-09 12:17:04|EUR / MWh|63.06|2026-01-09 00:45:00|
|2026-01-09 12:17:04|EUR / MWh|64.21|2026-01-09 01:00:00|
|2026-01-09 12:17:04|EUR / MWh|58.11|2026-01-09 01:15:00|
|2026-01-09 12:17:04|EUR / MWh|54.79|2026-01-09 01:30:00|
|2026-01-09 12:17:04|EUR / MWh|52.29|2026-01-09 01:45:00|
|2026-01-09 12:17:04|EUR / MWh|55.75|2026-01-09 02:00:00|
|2026-01-09 12:17:04|EUR / MWh|51.03|2026-01-09 02:15:00|
|2026-01-09 12